# Anaya Sharma — Wan2.1 I2V Fashion Reel Generator\n## Production Kaggle Notebook (v3.0)\n\n**Before running:**\n1. Add your `anaya-assets` dataset (contains outfit image + anaya_avatar.jpg)\n2. Set Accelerator → **GPU T4 x2**\n3. Set Internet → **ON**\n4. Add your HuggingFace token as a **Kaggle Secret** named `HF_TOKEN`\n   (Notebook sidebar → Add-ons → Secrets → Add New Secret)\n\n**This notebook generates REAL AI video using Wan2.1 Image-to-Video diffusion.**\nIt does NOT use pixel warping, Ken Burns zoom, or image crossfading.\nIf GPU inference fails for any reason, the notebook will report failure clearly.

In [ ]:
# CELL 1: GPU & Environment Check\nimport subprocess, sys, os, platform\n\nprint("=" * 60)\nprint("  ENVIRONMENT CHECK")\nprint("=" * 60)\n\nresult = subprocess.run(["nvidia-smi"], capture_output=True, text=True)\nif result.returncode != 0:\n    raise RuntimeError(\n        "No GPU detected. Enable GPU: Settings → Accelerator → GPU T4 x2"\n    )\nprint(result.stdout)\n\nimport torch\nprint(f"PyTorch     : {torch.__version__}")\nprint(f"CUDA avail  : {torch.cuda.is_available()}")\nif torch.cuda.is_available():\n    props = torch.cuda.get_device_properties(0)\n    total_vram_gb = props.total_memory / 1e9\n    print(f"GPU         : {props.name}")\n    print(f"VRAM        : {total_vram_gb:.1f} GB")\n    if total_vram_gb < 14:\n        print("WARNING: Less than 14 GB VRAM — will force 1.3B model.")\n    elif total_vram_gb < 18:\n        print("NOTE: ~16 GB VRAM — will use 14B with CPU offloading.")\n        print("      If OOM occurs, set USE_SMALL_MODEL = True in Cell 2.")\nprint("=" * 60)

In [ ]:
# CELL 2: Configuration  (single source of truth — matches pipeline_config.py)\n# ──────────────────────────────────────────────────────────────────────────────\n# Set USE_SMALL_MODEL = True if you get CUDA out-of-memory on T4.\n# The 1.3B model is faster and fits easily; quality is lower but still real AI motion.\nUSE_SMALL_MODEL = False\n\nimport os\nfrom pathlib import Path\n\n# ── Model ──────────────────────────────────────────────────────────────────\nMODEL_ID_14B  = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"\nMODEL_ID_1_3B = "Wan-AI/Wan2.1-I2V-1.3B-480P-Diffusers"\nMODEL_ID = MODEL_ID_1_3B if USE_SMALL_MODEL else MODEL_ID_14B\n\n# ── Resolution (Wan2.1 480P official supported dimensions) ─────────────────\nW, H = 480, 832      # 9:16 portrait — do NOT change to arbitrary values\n\n# ── Generation parameters ──────────────────────────────────────────────────\nNUM_FRAMES        = 81      # ~5.1 s at 16 fps\nOUTPUT_FPS        = 16\nGUIDANCE_SCALE    = 5.0     # 5.0 for 480P; higher values suppress motion\nNUM_STEPS         = 50      # 50 for quality; 30 if time-constrained\nSEED              = 42      # change to 123 / 777 if result is too static\n\n# ── Scheduler ──────────────────────────────────────────────────────────────\n# flow_shift MUST be applied via from_config(), not by mutating config dict.\nFLOW_SHIFT = 3.0   # 3.0 recommended for 480P\n\n# ── Paths ───────────────────────────────────────────────────────────────────\nASSET_DIR  = Path("/kaggle/input/anaya-assets")\nOUTPUT_DIR = Path("/kaggle/working")\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\n# Auto-detect if running on Colab instead\nif not ASSET_DIR.exists():\n    ASSET_DIR  = Path("/content")\n    OUTPUT_DIR = Path("/content/output")\n    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    print("Colab path detected.")\n\nAVATAR_PATH      = ASSET_DIR / "anaya_avatar.jpg"\nOUTPUT_VIDEO     = OUTPUT_DIR / "anaya_fashion_reel.mp4"\nCONTACT_SHEET    = OUTPUT_DIR / "contact_sheet.jpg"\nRUN_REPORT_PATH  = OUTPUT_DIR / "run_report.json"\n\n# ── Prompts ─────────────────────────────────────────────────────────────────\n# Describe TEMPORAL ACTION, not static appearance.\n# Motion verbs must come first. ~90 words is the effective range.\nMOTION_PROMPT = (\n    "An elegant young Indian woman presents her outfit in a natural full-body fashion shot. "\n    "She maintains a relaxed standing pose, gently shifts her weight once, makes a subtle "\n    "head movement, softly smiles, and lets the skirt and loose fabric respond naturally "\n    "to the small body motion. Her hair moves slightly with the motion. "\n    "The camera stays nearly fixed. Continuous realistic human motion, coherent anatomy, "\n    "natural lighting, realistic fabric drape and folds. "\n    "Preserve the exact outfit design, colour and pattern from the input image."\n)\n\nNEGATIVE_PROMPT = (\n    "frozen, static image, no movement, slideshow, Ken Burns zoom, artificial pan, "\n    "sudden pose change, camera cut, identity drift, facial flicker, face shape change, "\n    "clothing redesign, changing prints or colours, disappearing fabric, unstable hem, "\n    "extra limbs, missing limbs, deformed hands, floating body parts, feet sliding, "\n    "phone, props, additional people, text, logo, border, watermark, "\n    "excessive hair motion, implausible physics, blurry, low quality, cartoon, CGI"\n)\n\nprint(f"Model      : {MODEL_ID}")\nprint(f"Resolution : {W}x{H}  |  Frames: {NUM_FRAMES}  |  FPS: {OUTPUT_FPS}")\nprint(f"Guidance   : {GUIDANCE_SCALE}  |  Steps: {NUM_STEPS}  |  Seed: {SEED}")\nprint(f"Flow shift : {FLOW_SHIFT}  (applied via scheduler.from_config)")\nprint(f"Asset dir  : {ASSET_DIR}  (exists: {ASSET_DIR.exists()})")\nprint(f"Avatar     : {AVATAR_PATH}  (exists: {AVATAR_PATH.exists()})")\nprint(f"Output dir : {OUTPUT_DIR}")\nprint()\nprint(f"Prompt ({len(MOTION_PROMPT.split())} words):")\nprint(MOTION_PROMPT)

In [ ]:
# CELL 3: Install Dependencies  (~3-5 minutes on first run)\nimport subprocess\n\ndef run(cmd):\n    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)\n    if r.returncode != 0:\n        print("STDERR:", r.stderr[-2000:])\n    return r.returncode == 0\n\nprint("Installing diffusers and video stack...")\nassert run("pip install -q --upgrade diffusers transformers accelerate sentencepiece")\nassert run("pip install -q imageio imageio-ffmpeg pillow numpy scipy opencv-python-headless")\nprint("All dependencies installed.")

In [ ]:
# CELL 4: HuggingFace Authentication\n# Reads HF_TOKEN from Kaggle Secrets (recommended) or environment variable.\n# Never hardcode your token here.\nfrom huggingface_hub import login\nimport os\n\n# Try Kaggle secret first, then env var\ntry:\n    from kaggle_secrets import UserSecretsClient\n    hf_token = UserSecretsClient().get_secret("HF_TOKEN")\n    print("HF_TOKEN loaded from Kaggle Secrets.")\nexcept Exception:\n    hf_token = os.environ.get("HF_TOKEN", "")\n    if hf_token:\n        print("HF_TOKEN loaded from environment variable.")\n    else:\n        raise RuntimeError(\n            "HF_TOKEN not found!\n"\n            "Add it: Notebook sidebar → Add-ons → Secrets → HF_TOKEN"\n        )\n\nlogin(token=hf_token, add_to_git_credential=False)\nprint("HuggingFace authentication: OK")

In [ ]:
# CELL 5: Load Wan2.1 I2V Model\n# Applies the correct flow_shift via scheduler.from_config() — NOT dict mutation.\nimport torch\nimport gc\nfrom diffusers import WanImageToVideoPipeline, FlowMatchEulerDiscreteScheduler\n\nprint(f"Loading model: {MODEL_ID}")\nprint("First run downloads ~15-30 GB. Subsequent runs use cache.")\nprint("This may take 5-15 minutes...")\n\ndtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n\ntry:\n    pipe = WanImageToVideoPipeline.from_pretrained(\n        MODEL_ID,\n        torch_dtype=dtype,\n        token=hf_token,\n    )\nexcept Exception as e:\n    raise RuntimeError(\n        f"Model load failed: {e}\n"\n        f"If authentication error: verify HF_TOKEN has read access to {MODEL_ID}\n"\n        f"If disk/memory error: set USE_SMALL_MODEL = True in Cell 2"\n    )\n\n# ── Apply flow_shift correctly ──────────────────────────────────────────────\n# The ONLY correct way: reconstruct scheduler via from_config with shift param.\n# Mutating pipe.scheduler.config['shift'] after-the-fact does NOT work reliably.\ntry:\n    pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(\n        pipe.scheduler.config,\n        shift=FLOW_SHIFT,\n    )\n    print(f"Scheduler: FlowMatchEulerDiscreteScheduler  flow_shift={FLOW_SHIFT}")\nexcept Exception as e:\n    print(f"Warning: Could not set flow_shift ({e}). Proceeding with default scheduler.")\n\n# ── Memory optimisations (critical for T4 16 GB) ───────────────────────────\npipe.enable_model_cpu_offload()   # offloads unused layers to CPU RAM\npipe.enable_vae_slicing()          # decode VAE slice-by-slice to save VRAM\n\ngc.collect()\ntorch.cuda.empty_cache()\n\nvram_used  = round(torch.cuda.memory_allocated() / 1e9, 2)\nvram_total = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)\nprint(f"Model loaded. VRAM used: {vram_used} / {vram_total} GB")

In [ ]:
# CELL 6: Prepare Conditioning Image (letterbox — preserves full outfit)\nimport glob\nfrom PIL import Image\nimport numpy as np\n\n# Find dress images; fall back to avatar if none found\ndress_paths = sorted(glob.glob(str(ASSET_DIR / "lakdata_Women_Tiered_Pur_hd_angle_*.jpg")))\nprint(f"Found {len(dress_paths)} dress reference images.")\nfor p in dress_paths:\n    print(f"  {p}")\n\nPRIMARY_PATH = dress_paths[0] if dress_paths else str(AVATAR_PATH)\nprint(f"\nUsing conditioning image: {PRIMARY_PATH}")\n\nraw = Image.open(PRIMARY_PATH).convert("RGB")\norig_w, orig_h = raw.size\nprint(f"Original size: {orig_w}x{orig_h}")\n\n# Letterbox: scale to fit WxH while preserving aspect ratio\n# Never stretch — squishing distorts body proportions and confuses the model\nscale = min(W / orig_w, H / orig_h)\nnew_w, new_h = int(orig_w * scale), int(orig_h * scale)\nresized = raw.resize((new_w, new_h), Image.Resampling.LANCZOS)\n\n# Neutral warm background — avoids stark black bars that confuse diffusion\ncanvas = Image.new("RGB", (W, H), (245, 240, 235))\npaste_x = (W - new_w) // 2\npaste_y = (H - new_h) // 2\ncanvas.paste(resized, (paste_x, paste_y))\nconditioning_image = canvas\n\ncanvas.save(str(OUTPUT_DIR / "conditioning_frame.jpg"))\nprint(f"Conditioning image prepared: {W}x{H} (letterboxed, full outfit preserved)")

In [ ]:
# CELL 7: Generate Video — Real Wan2.1 I2V Diffusion\n# ─────────────────────────────────────────────────────────────────────────────\n# This cell runs the actual neural network inference.\n# Expected time: 15-30 min on T4 (14B), 5-10 min (1.3B)\n# If CUDA OOM: set USE_SMALL_MODEL = True in Cell 2 and re-run from Cell 5.\n# ─────────────────────────────────────────────────────────────────────────────\nimport torch, time\n\nprint("Starting Wan2.1 I2V generation...")\nprint(f"  Frames : {NUM_FRAMES}  |  Steps : {NUM_STEPS}  |  Guidance : {GUIDANCE_SCALE}")\nprint(f"  Seed   : {SEED}")\nprint(f"  Prompt : {MOTION_PROMPT[:120]}...")\nprint()\n\ngeneration_start = time.time()\ngeneration_error = None\nraw_frames = None\n\ntry:\n    with torch.inference_mode():\n        output = pipe(\n            image=conditioning_image,\n            prompt=MOTION_PROMPT,\n            negative_prompt=NEGATIVE_PROMPT,\n            height=H,\n            width=W,\n            num_frames=NUM_FRAMES,\n            guidance_scale=GUIDANCE_SCALE,\n            num_inference_steps=NUM_STEPS,\n            generator=torch.Generator("cuda").manual_seed(SEED),\n        )\n    raw_frames = output.frames[0]  # list of PIL Images\n    generation_time = round(time.time() - generation_start, 1)\n    print(f"Generation complete in {generation_time}s — {len(raw_frames)} frames produced.")\n\nexcept torch.cuda.OutOfMemoryError as e:\n    generation_error = f"CUDA OOM: {e}"\n    print(f"\nOUT OF MEMORY ERROR: {generation_error}")\n    print("Fix: set USE_SMALL_MODEL = True in Cell 2, then re-run from Cell 5.")\n    raise\n\nexcept Exception as e:\n    generation_error = str(e)\n    print(f"\nGENERATION FAILED: {generation_error}")\n    raise

In [ ]:
# CELL 8: Quality Validation\n# Distinguishes too-little-motion from plausible motion from excessive drift.\n# These checks are DIAGNOSTIC — they do not guarantee perceptual quality.\nimport numpy as np\n\nif raw_frames is None:\n    raise RuntimeError("No frames to validate — generation did not complete.")\n\nframes_np = [np.array(f).astype(np.float32) for f in raw_frames]\n\n# ── 1. Motion score: mean absolute pixel difference first→last ────────────\nf0, f_last = frames_np[0], frames_np[-1]\nmotion_score = float(np.mean(np.abs(f_last - f0)))\nprint(f"Motion score (first→last frame diff): {motion_score:.2f}")\nprint(f"  < {2.0} = likely frozen/slideshow")\nprint(f"  {2.0}–{40.0} = plausible subject movement")\nprint(f"  > {40.0} = possible identity drift or camera jitter")\n\n# ── 2. Temporal consistency: frame-to-frame mean diff ────────────────────\nframe_diffs = [\n    float(np.mean(np.abs(frames_np[i+1] - frames_np[i])))\n    for i in range(0, len(frames_np) - 1, 5)   # sample every 5 frames\n]\nmean_jitter = float(np.mean(frame_diffs))\nmax_jitter  = float(np.max(frame_diffs))\nprint(f"\nTemporal consistency (sampled every 5 frames):")\nprint(f"  Mean frame-to-frame diff : {mean_jitter:.2f}")\nprint(f"  Max  frame-to-frame diff : {max_jitter:.2f}")\nif max_jitter > 30:\n    print("  WARNING: High max jitter — may indicate a scene cut or identity flip.")\n\n# ── 3. Determine review status ────────────────────────────────────────────\nif motion_score < 2.0:\n    review_status = "NEEDS_REVIEW__TOO_STATIC"\n    print(f"\nStatus: {review_status}")\n    print("  Retry: change SEED to 123 or 777, lower GUIDANCE_SCALE to 4.0")\nelif motion_score > 40.0:\n    review_status = "NEEDS_REVIEW__EXCESSIVE_MOTION"\n    print(f"\nStatus: {review_status}")\n    print("  Retry: raise GUIDANCE_SCALE to 6.0, change SEED")\nelse:\n    review_status = "MOTION_OK__REVIEW_VISUALLY"\n    print(f"\nStatus: {review_status}")\n    print("  Motion is in the plausible range. Review the contact sheet below.")\n\nprint("\n  *** Always review the video manually before publishing. ***")

In [ ]:
# CELL 9: Export Clean MP4\nimport imageio, numpy as np\n\nif raw_frames is None:\n    raise RuntimeError("No frames — cannot export. Fix generation errors first.")\n\nprint(f"Exporting {len(raw_frames)} frames to {OUTPUT_VIDEO} ...")\n\nwriter = imageio.get_writer(\n    str(OUTPUT_VIDEO),\n    fps=OUTPUT_FPS,\n    codec="libx264",\n    output_params=["-pix_fmt", "yuv420p", "-crf", "18"],\n)\nfor frame in raw_frames:\n    writer.append_data(np.array(frame))\nwriter.close()\n\nimport os\nsize_mb = round(os.path.getsize(str(OUTPUT_VIDEO)) / 1e6, 2)\nprint(f"Exported: {OUTPUT_VIDEO}  ({size_mb} MB)")\nprint("No text overlays, no watermarks, no borders — clean 9:16 footage only.")

In [ ]:
# CELL 10: Save Contact Sheet (representative frames for visual review)\nimport numpy as np\nfrom PIL import Image\n\nSTRIDE = 10   # one frame every STRIDE frames\nsampled = raw_frames[::STRIDE]\nprint(f"Contact sheet: {len(sampled)} frames sampled every {STRIDE} frames")\n\ncols = min(4, len(sampled))\nrows = (len(sampled) + cols - 1) // cols\nthumb_w, thumb_h = W // 2, H // 2\n\nsheet = Image.new("RGB", (cols * thumb_w, rows * thumb_h), (30, 30, 30))\nfor idx, frame in enumerate(sampled):\n    thumb = frame.resize((thumb_w, thumb_h), Image.Resampling.LANCZOS)\n    r, c = divmod(idx, cols)\n    sheet.paste(thumb, (c * thumb_w, r * thumb_h))\n\nsheet.save(str(CONTACT_SHEET))\nprint(f"Contact sheet saved: {CONTACT_SHEET}")\n\n# Display inline\nfrom IPython.display import display\ndisplay(sheet.resize((sheet.width // 2, sheet.height // 2)))

In [ ]:
# CELL 11: Save Run Report JSON\nimport json, datetime, torch\n\nreport = {\n    "schema_version": "3.0",\n    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),\n    "model": {\n        "model_id": MODEL_ID,\n        "dtype": str(dtype),\n    },\n    "scheduler": {\n        "class": "FlowMatchEulerDiscreteScheduler",\n        "flow_shift": FLOW_SHIFT,\n    },\n    "generation": {\n        "prompt_word_count": len(MOTION_PROMPT.split()),\n        "width": W,\n        "height": H,\n        "num_frames": NUM_FRAMES,\n        "fps": OUTPUT_FPS,\n        "guidance_scale": GUIDANCE_SCALE,\n        "num_inference_steps": NUM_STEPS,\n        "seed": SEED,\n        "generation_time_s": generation_time if 'generation_time' in dir() else None,\n    },\n    "hardware": {\n        "gpu_name": torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else "N/A",\n        "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0,\n    },\n    "quality": {\n        "motion_score": round(motion_score, 3) if 'motion_score' in dir() else None,\n        "mean_jitter": round(mean_jitter, 3) if 'mean_jitter' in dir() else None,\n        "max_jitter": round(max_jitter, 3) if 'max_jitter' in dir() else None,\n        "review_status": review_status if 'review_status' in dir() else "UNKNOWN",\n    },\n    "output": {\n        "video": str(OUTPUT_VIDEO),\n        "contact_sheet": str(CONTACT_SHEET),\n    },\n    "generation_method": "WAN2.1_I2V_DIFFUSION__GENUINE_AI_VIDEO",\n}\n\nwith open(str(RUN_REPORT_PATH), "w") as f:\n    json.dump(report, f, indent=2)\n\nprint("Run report saved:", RUN_REPORT_PATH)\nprint()\nprint(json.dumps(report, indent=2))

In [ ]:
# CELL 12: Display Output\nfrom IPython.display import Video, display\n\nprint("Final output video:")\nprint(f"  {OUTPUT_VIDEO}")\nprint()\nprint("Download from the Output tab (right sidebar) when notebook finishes.")\nprint()\n# Inline preview (may not work in all Kaggle versions)\ntry:\n    display(Video(str(OUTPUT_VIDEO), width=320))\nexcept Exception:\n    print("(Inline preview not available — download from Output tab)")